# RAG Experimentation Notebook

This notebook implements and tests the complete RAG (Retrieval-Augmented Generation) pipeline for the Yoga Assistant system.

## Objectives

1. Load best retrieval system from experiments (Hybrid Search)
2. Implement RAG flow: retrieval → context assembly → LLM generation
3. Test with sample questions
4. Experiment with multiple LLM models


## Setup and Imports


In [2]:
import pandas as pd
import numpy as np
import os
import sys
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
import warnings
import time

warnings.filterwarnings("ignore")

# Add parent directory to path for imports
sys.path.append("..")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

## Load Environment Variables


In [3]:
# Load environment variables from .env file
load_dotenv()

# Get API configuration
HYPERBOLIC_API_KEY = os.getenv("HYPERBOLIC_API_KEY")
LLM_MODEL = os.getenv("LLM_MODEL", "meta-llama/Meta-Llama-3.1-70B-Instruct")

print(f"API Key loaded: {'✓' if HYPERBOLIC_API_KEY else '✗'}")
print(f"Default LLM Model: {LLM_MODEL}")

API Key loaded: ✗
Default LLM Model: meta-llama/Meta-Llama-3.1-70B-Instruct


## Set Up LLM API Connection

We'll use the OpenAI-compatible API format that works with both Hyperbolic and Nebius.


In [4]:
from openai import OpenAI

api_key = os.getenv("LLM_API_KEY")
base_url = os.getenv("LLM_BASE_URL", "https://api.hyperbolic.xyz/v1")

# Initialize OpenAI client with Hyperbolic endpoint
client = OpenAI(api_key=api_key, base_url=base_url)

print("✓ LLM API client initialized")

✓ LLM API client initialized


## Test LLM Connection


In [15]:
def test_llm_connection(model: str = LLM_MODEL) -> bool:
    """Test if LLM API is working correctly."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": "Say 'Hello' if you can hear me."}
            ],
            max_tokens=50,
            temperature=0.0,
        )

        answer = response.choices[0].message.content
        print(f"✓ LLM Response: {answer}")
        print(f"✓ Model: {model}")
        print(f"✓ Tokens used: {response.usage.total_tokens}")
        return True
    except Exception as e:
        print(f"✗ Error: {e}")
        return False


# Test the connection
test_llm_connection()

✓ LLM Response: Hello.
✓ Model: meta-llama/Meta-Llama-3.1-70B-Instruct
✓ Tokens used: 48


True

## Load Yoga Data


In [5]:
# Load yoga poses dataset
yoga_data = pd.read_csv("../data/yoga_data_merged.csv")
print(f"Loaded {len(yoga_data)} yoga poses")

# Create pose dictionary for quick lookup
pose_dict = {row["id"]: row.to_dict() for _, row in yoga_data.iterrows()}
print(f"Created pose dictionary with {len(pose_dict)} poses")

Loaded 202 yoga poses
Created pose dictionary with 202 poses


## Load Best Retrieval System

We'll use the best retrieval approach from our experiments (notebook 03):

- **Weighted Product Hybrid Search** (BM25 + Vector)
- **Alpha = 0.4** (40% BM25, 60% Vector)
- **Performance**: 76% Hit Rate, 66% MRR


In [6]:
from yoga_assistant.retrieval import create_retrieval_system

# Create the retrieval system with best configuration
print("Initializing retrieval system...\n")
retrieval_system = create_retrieval_system(pose_dict)

# Test retrieval
test_query = "What poses help with balance?"
retrieved_ids = retrieval_system.search(test_query, top_k=5)

print(f"\nTest query: '{test_query}'")
print(f"Retrieved {len(retrieved_ids)} poses:")
for i, pose_id in enumerate(retrieved_ids, 1):
    pose = pose_dict[pose_id]
    print(f"{i}. {pose['pose_name']} (ID: {pose_id}, {pose['category']})")

Initializing retrieval system...

Creating retrieval system with best configuration...
- BM25: all 7 fields
- Vector: all-mpnet-base-v2
- Hybrid: Weighted Product (alpha=0.4)

Loading embedding model: all-mpnet-base-v2...
Model loaded. Embedding dimension: 768
Generating embeddings for 202 poses...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings generated. Shape: (202, 768)
✓ Retrieval system ready!

Test query: 'What poses help with balance?'
Retrieved 5 poses:
1. Cat-Cow Pose (ID: 5, standing)
2. Warrior II on One Leg (ID: 118, balancing)
3. Peacock Pose (ID: 33, balancing)
4. River Rock (ID: 199, balancing)
5. Warrior Pose (ID: 109, standing)


## Implement Context Assembly

Convert retrieved poses into a formatted context string for the LLM.


In [7]:
def assemble_context(retrieved_pose_ids: List[int]) -> str:
    """
    Format retrieved poses into a context string for the LLM.

    Args:
        retrieved_pose_ids: List of pose IDs

    Returns:
        Formatted context string
    """
    if not retrieved_pose_ids:
        return "No relevant yoga poses found."

    context_parts = []

    for i, pose_id in enumerate(retrieved_pose_ids, 1):
        pose = pose_dict[pose_id]
        context = f"""Pose {i}: {pose['pose_name']} ({pose['sanskrit_name']})
Category: {pose['category']}
Difficulty: {pose['difficulty_level']}
Benefits: {pose['benefits']}
Contraindications: {pose['contraindications']}
Instructions: {pose['instructions']}
Modifications: {pose['modifications']}
"""
        context_parts.append(context)

    return "\n---\n".join(context_parts)


# Test context assembly
test_context = assemble_context(retrieved_ids[:2])
print("Example context (first 500 chars):")
print(test_context[:500] + "...")

Example context (first 500 chars):
Pose 1: Cat-Cow Pose (Marjaryasana-Bitilasana)
Category: standing
Difficulty: beginner
Benefits: The Cat-Cow Pose stretches the spine, neck, and torso, while also improving flexibility and reducing tension. This pose can also help to warm up the body and prepare it for more dynamic movements. Additionally, it can help to calm the mind and promote relaxation.
Contraindications: This pose is generally safe for most people, but those with severe neck injuries or cervical spine problems should avoid...


## Create Prompt Template


In [8]:
def create_prompt(question: str, context: str) -> str:
    """
    Create a prompt for the LLM with question and context.

    Args:
        question: User's question
        context: Retrieved yoga pose information

    Returns:
        Formatted prompt string
    """
    prompt = f"""You are a knowledgeable yoga instructor assistant. Answer the user's question based ONLY on the provided yoga pose information. Be accurate, helpful, and concise.

If the provided information doesn't contain the answer, say so politely and suggest the user rephrase their question.

YOGA POSE INFORMATION:
{context}

USER QUESTION:
{question}

ANSWER:"""

    return prompt


# Test prompt creation
test_prompt = create_prompt(test_query, test_context[:500])
print("Example prompt (first 600 chars):")
print(test_prompt[:600] + "...")

Example prompt (first 600 chars):
You are a knowledgeable yoga instructor assistant. Answer the user's question based ONLY on the provided yoga pose information. Be accurate, helpful, and concise.

If the provided information doesn't contain the answer, say so politely and suggest the user rephrase their question.

YOGA POSE INFORMATION:
Pose 1: Cat-Cow Pose (Marjaryasana-Bitilasana)
Category: standing
Difficulty: beginner
Benefits: The Cat-Cow Pose stretches the spine, neck, and torso, while also improving flexibility and reducing tension. This pose can also help to warm up the body and prepare it for more dynamic movements. ...


## Implement Complete RAG Pipeline


In [9]:
def rag_pipeline(
    question: str,
    model: str = LLM_MODEL,
    top_k: int = 5,
    temperature: float = 0.3,
    max_tokens: int = 500,
) -> Dict[str, Any]:
    """
    Complete RAG pipeline: retrieve → assemble context → generate answer.

    Args:
        question: User's question
        model: LLM model to use
        top_k: Number of poses to retrieve
        temperature: LLM temperature (0.0 = deterministic, 1.0 = creative)
        max_tokens: Maximum tokens in response

    Returns:
        Dictionary with answer, retrieved_poses, tokens_used, response_time
    """
    start_time = time.time()

    # Step 1: Retrieve relevant poses using hybrid search
    retrieved_ids = retrieval_system.search(question, top_k=top_k)

    if not retrieved_ids:
        return {
            "answer": "I couldn't find any relevant yoga poses for your question. Could you please rephrase or ask about a specific pose, category, or benefit?",
            "retrieved_poses": [],
            "tokens_used": 0,
            "response_time_ms": int((time.time() - start_time) * 1000),
            "model": model,
        }

    # Step 2: Assemble context
    context = assemble_context(retrieved_ids)

    # Step 3: Create prompt
    prompt = create_prompt(question, context)

    # Step 4: Call LLM
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
        )

        answer = response.choices[0].message.content
        tokens_used = response.usage.total_tokens

    except Exception as e:
        answer = f"Error generating response: {str(e)}"
        tokens_used = 0

    response_time_ms = int((time.time() - start_time) * 1000)

    return {
        "answer": answer,
        "retrieved_poses": [
            {
                "id": pose_id,
                "pose_name": pose_dict[pose_id]["pose_name"],
                "category": pose_dict[pose_id]["category"],
                "difficulty_level": pose_dict[pose_id]["difficulty_level"],
            }
            for pose_id in retrieved_ids
        ],
        "tokens_used": tokens_used,
        "response_time_ms": response_time_ms,
        "model": model,
    }


print("✓ RAG pipeline implemented")

✓ RAG pipeline implemented


## Test RAG Pipeline with Sample Questions


In [10]:
# Test with a sample question
test_question = "What poses help with balance?"

print(f"Question: {test_question}\n")
result = rag_pipeline(test_question)

print(f"Answer:\n{result['answer']}\n")
print(f"Retrieved Poses:")
for pose in result["retrieved_poses"]:
    print(
        f"  - {pose['pose_name']} (ID: {pose['id']}, {pose['difficulty_level']})"
    )
print(f"\nTokens Used: {result['tokens_used']}")
print(f"Response Time: {result['response_time_ms']}ms")
print(f"Model: {result['model']}")

Question: What poses help with balance?

Answer:
According to the provided yoga pose information, the following poses can help with balance:

1. Warrior II on One Leg (Eka Pada Virabhadrasana II) - This pose strengthens the ankles and legs, while also improving balance and focus.
2. Peacock Pose (Mayurasana) - This pose improves balance and overall core stability.
3. River Rock (Nadi Shila) - This pose improves balance and focus, while also engaging the core and strengthening the ankles.

Additionally, Warrior Pose (Virabhadrasana) can also help with balance and stability, although it's more focused on strengthening the legs, hips, and core.

Retrieved Poses:
  - Cat-Cow Pose (ID: 5, beginner)
  - Warrior II on One Leg (ID: 118, intermediate)
  - Peacock Pose (ID: 33, intermediate)
  - River Rock (ID: 199, advanced)
  - Warrior Pose (ID: 109, beginner)

Tokens Used: 1679
Response Time: 2521ms
Model: meta-llama/Meta-Llama-3.1-70B-Instruct


In [11]:
# Test with more sample questions
sample_questions = [
    "What are the benefits of Cobra Pose?",
    "Can you recommend beginner standing poses?",
    "What poses should I avoid if I have a back injury?",
    "How do I do Tree Pose?",
]

print("Testing RAG pipeline with multiple questions:\n")
print("=" * 80)

for i, question in enumerate(sample_questions, 1):
    print(f"\n{i}. Question: {question}")
    result = rag_pipeline(question)
    print(
        f"\nAnswer: {result['answer'][:300]}..."
        if len(result["answer"]) > 300
        else f"\nAnswer: {result['answer']}"
    )
    print(
        f"Retrieved: {len(result['retrieved_poses'])} poses | Tokens: {result['tokens_used']} | Time: {result['response_time_ms']}ms"
    )
    print("=" * 80)

Testing RAG pipeline with multiple questions:


1. Question: What are the benefits of Cobra Pose?

Answer: The benefits of Cobra Pose (Bhujangasana) include strengthening the back muscles, opening the chest, and improving flexibility in the shoulders and upper back. It also helps to relieve stress and anxiety, promoting a sense of calm and relaxation. Regular practice of this pose can also improve breath...
Retrieved: 5 poses | Tokens: 1776 | Time: 1112ms

2. Question: Can you recommend beginner standing poses?

Answer: Based on the provided yoga pose information, I can recommend the following beginner standing poses:

1. Warrior Pose I (Virabhadrasana I) - This pose strengthens the legs, hips, and core, while also stretching the chest and shoulders.
2. Warrior Pose (Virabhadrasana) - This pose strengthens the legs...
Retrieved: 5 poses | Tokens: 1732 | Time: 1396ms

3. Question: What poses should I avoid if I have a back injury?

Answer: Based on the provided yoga pose information, if

## Query Rewriting Experiments

Query rewriting uses an LLM to enhance or clarify user questions before retrieval. This can improve retrieval quality by:

- Expanding abbreviations or unclear terms
- Adding context or yoga-specific terminology
- Reformulating vague questions into more specific queries

We'll test query rewriting and measure its impact on:

- Retrieval quality (hit rate, MRR)
- Answer quality (subjective evaluation)
- Cost and latency


### Load Ground Truth for Evaluation


In [12]:
# Load ground truth dataset for evaluation
ground_truth = pd.read_csv("../data/ground_truth.csv")

In [13]:
# Parse relevant_pose_ids from string to list of integers
def parse_pose_ids(pose_ids_str: str) -> List[int]:
    """Convert comma-separated string of pose IDs to list of integers."""
    if pd.isna(pose_ids_str):
        return []
    return [int(x.strip()) for x in str(pose_ids_str).split(",")]

In [14]:
ground_truth["relevant_pose_ids_list"] = ground_truth[
    "relevant_pose_ids"
].apply(parse_pose_ids)

print(f"Loaded {len(ground_truth)} ground truth questions")
print(f"\nSample questions:")
for i, row in ground_truth.head(5).iterrows():
    print(f"{i+1}. {row['question']}")

Loaded 75 ground truth questions

Sample questions:
1. What pose is great for improving balance and focus?
2. What are the benefits of Cobra Pose?
3. Can you give me some standing poses that are good for beginners?
4. How do I do a Cat-Cow Pose?
5. What poses should I avoid if I have a recent back injury?


### Implement Evaluation Metrics


In [15]:
def calculate_hit_rate(
    retrieved_ids: List[List[int]], relevant_ids: List[List[int]]
) -> float:
    """Calculate hit rate: percentage of queries where at least one relevant document is retrieved."""
    hits = 0
    for retrieved, relevant in zip(retrieved_ids, relevant_ids):
        if any(doc_id in relevant for doc_id in retrieved):
            hits += 1
    return hits / len(retrieved_ids) if len(retrieved_ids) > 0 else 0.0


def calculate_mrr(
    retrieved_ids: List[List[int]], relevant_ids: List[List[int]]
) -> float:
    """Calculate Mean Reciprocal Rank (MRR): average of 1/rank of first relevant document."""
    reciprocal_ranks = []
    for retrieved, relevant in zip(retrieved_ids, relevant_ids):
        for rank, doc_id in enumerate(retrieved, start=1):
            if doc_id in relevant:
                reciprocal_ranks.append(1.0 / rank)
                break
        else:
            reciprocal_ranks.append(0.0)
    return np.mean(reciprocal_ranks) if len(reciprocal_ranks) > 0 else 0.0


print("✓ Evaluation metrics implemented")

✓ Evaluation metrics implemented


### Implement Query Rewriting


In [16]:
def rewrite_query(question: str, model: str = LLM_MODEL) -> str:
    """
    Use LLM to rewrite/enhance user query for better retrieval.

    Args:
        question: Original user question
        model: LLM model to use

    Returns:
        Rewritten query
    """
    rewrite_prompt = f"""You are a yoga expert assistant. Your task is to rewrite the user's question to make it more specific and suitable for searching a yoga pose database.

Rules:
- Keep the core intent of the question
- Add relevant yoga terminology if appropriate
- Expand abbreviations or unclear terms
- Make vague questions more specific
- Keep it concise (1-2 sentences max)
- If the question is already clear and specific, return it unchanged

Original question: {question}

Rewritten question:"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": rewrite_prompt}],
            max_tokens=100,
            temperature=0.3,
        )
        rewritten = response.choices[0].message.content.strip()
        return rewritten
    except Exception as e:
        print(f"Error rewriting query: {e}")
        return question  # Fall back to original question


print("✓ Query rewriting function implemented")

✓ Query rewriting function implemented


### Test Query Rewriting with Examples


In [17]:
# Test query rewriting with sample questions
test_questions = [
    "What poses help with balance?",
    "I have back pain",
    "Show me beginner poses",
    "What's good for flexibility?",
    "Poses for stress",
]

print("Testing Query Rewriting:\n")
print("=" * 80)

for question in test_questions:
    rewritten = rewrite_query(question)
    print(f"\nOriginal:  {question}")
    print(f"Rewritten: {rewritten}")
    print("-" * 80)

Testing Query Rewriting:


Original:  What poses help with balance?
Rewritten: What yoga asanas improve equilibrium and stability, specifically targeting balance and focus?
--------------------------------------------------------------------------------

Original:  I have back pain
Rewritten: What are some gentle yoga poses that can help alleviate lower back pain and promote spinal flexibility?
--------------------------------------------------------------------------------

Original:  Show me beginner poses
Rewritten: Show me foundational yoga poses suitable for beginners, focusing on gentle stretches and basic postures.
--------------------------------------------------------------------------------

Original:  What's good for flexibility?
Rewritten: What yoga poses are beneficial for increasing overall flexibility, particularly in the hamstrings, hips, and lower back?
--------------------------------------------------------------------------------

Original:  Poses for stress
Rewrit

### Implement RAG Pipeline with Optional Query Rewriting


In [18]:
def rag_pipeline_with_rewriting(
    question: str,
    use_rewriting: bool = False,
    model: str = LLM_MODEL,
    top_k: int = 5,
    temperature: float = 0.3,
    max_tokens: int = 500,
) -> Dict[str, Any]:
    """
    RAG pipeline with optional query rewriting.

    Args:
        question: User's question
        use_rewriting: Whether to rewrite the query before retrieval
        model: LLM model to use
        top_k: Number of poses to retrieve
        temperature: LLM temperature
        max_tokens: Maximum tokens in response

    Returns:
        Dictionary with answer, retrieved_poses, tokens_used, response_time, etc.
    """
    start_time = time.time()

    # Step 1: Optional query rewriting
    search_query = question
    if use_rewriting:
        search_query = rewrite_query(question, model)

    # Step 2: Retrieve relevant poses
    retrieved_ids = retrieval_system.search(search_query, top_k=top_k)

    if not retrieved_ids:
        return {
            "answer": "I couldn't find any relevant yoga poses for your question. Could you please rephrase or ask about a specific pose, category, or benefit?",
            "retrieved_poses": [],
            "retrieved_ids": [],
            "tokens_used": 0,
            "response_time_ms": int((time.time() - start_time) * 1000),
            "model": model,
            "original_query": question,
            "search_query": search_query,
            "used_rewriting": use_rewriting,
        }

    # Step 3: Assemble context
    context = assemble_context(retrieved_ids)

    # Step 4: Create prompt
    prompt = create_prompt(
        question, context
    )  # Use original question for answer

    # Step 5: Call LLM
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        answer = response.choices[0].message.content
        tokens_used = response.usage.total_tokens
    except Exception as e:
        answer = f"Error generating response: {str(e)}"
        tokens_used = 0

    response_time_ms = int((time.time() - start_time) * 1000)

    return {
        "answer": answer,
        "retrieved_poses": [
            {
                "id": pose_id,
                "pose_name": pose_dict[pose_id]["pose_name"],
                "category": pose_dict[pose_id]["category"],
                "difficulty_level": pose_dict[pose_id]["difficulty_level"],
            }
            for pose_id in retrieved_ids
        ],
        "retrieved_ids": retrieved_ids,
        "tokens_used": tokens_used,
        "response_time_ms": response_time_ms,
        "model": model,
        "original_query": question,
        "search_query": search_query,
        "used_rewriting": use_rewriting,
    }


print("✓ RAG pipeline with query rewriting implemented")

✓ RAG pipeline with query rewriting implemented


### Compare Retrieval Quality: With vs Without Query Rewriting


In [19]:
# Evaluate on ground truth dataset
print("Evaluating retrieval quality with and without query rewriting...\n")

# Sample subset for faster evaluation (use all for final evaluation)
sample_size = 30  # Use 30 questions for quick evaluation
eval_data = ground_truth.head(sample_size)

# Without query rewriting
print("Running WITHOUT query rewriting...")
retrieved_without_rewriting = []
for _, row in eval_data.iterrows():
    result = rag_pipeline_with_rewriting(
        row["question"], use_rewriting=False, top_k=5
    )
    retrieved_without_rewriting.append(result["retrieved_ids"])

# With query rewriting
print("Running WITH query rewriting...")
retrieved_with_rewriting = []
for _, row in eval_data.iterrows():
    result = rag_pipeline_with_rewriting(
        row["question"], use_rewriting=True, top_k=5
    )
    retrieved_with_rewriting.append(result["retrieved_ids"])

# Calculate metrics
relevant_ids = eval_data["relevant_pose_ids_list"].tolist()

metrics_without = {
    "hit_rate": calculate_hit_rate(retrieved_without_rewriting, relevant_ids),
    "mrr": calculate_mrr(retrieved_without_rewriting, relevant_ids),
}

metrics_with = {
    "hit_rate": calculate_hit_rate(retrieved_with_rewriting, relevant_ids),
    "mrr": calculate_mrr(retrieved_with_rewriting, relevant_ids),
}

print("\n" + "=" * 80)
print("RETRIEVAL QUALITY COMPARISON")
print("=" * 80)
print(f"\nEvaluated on {len(eval_data)} questions\n")

print("WITHOUT Query Rewriting:")
print(f"  Hit Rate: {metrics_without['hit_rate']:.2%}")
print(f"  MRR:      {metrics_without['mrr']:.2%}")

print("\nWITH Query Rewriting:")
print(f"  Hit Rate: {metrics_with['hit_rate']:.2%}")
print(f"  MRR:      {metrics_with['mrr']:.2%}")

print("\nChange:")
hit_rate_change = metrics_with["hit_rate"] - metrics_without["hit_rate"]
mrr_change = metrics_with["mrr"] - metrics_without["mrr"]

if hit_rate_change > 0:
    print(f"  Hit Rate: {hit_rate_change:+.2%} (IMPROVEMENT ✓)")
else:
    print(f"  Hit Rate: {hit_rate_change:+.2%} (DEGRADATION ✗)")

if mrr_change > 0:
    print(f"  MRR:      {mrr_change:+.2%} (IMPROVEMENT ✓)")
else:
    print(f"  MRR:      {mrr_change:+.2%} (DEGRADATION ✗)")
print("=" * 80)

Evaluating retrieval quality with and without query rewriting...

Running WITHOUT query rewriting...
Running WITH query rewriting...

RETRIEVAL QUALITY COMPARISON

Evaluated on 30 questions

WITHOUT Query Rewriting:
  Hit Rate: 83.33%
  MRR:      75.28%

WITH Query Rewriting:
  Hit Rate: 66.67%
  MRR:      54.17%

Improvement:
  Hit Rate: -16.67% (improvement)
  MRR:      -21.11% (improvement)


### Side-by-Side Comparison: Sample Questions


In [20]:
# Compare answers for sample questions
comparison_questions = [
    "What poses help with balance?",
    "I have back pain",
    "Show me beginner poses",
]

print("\nSIDE-BY-SIDE COMPARISON\n")
print("=" * 80)

for i, question in enumerate(comparison_questions, 1):
    print(f"\n{i}. Question: {question}")
    print("-" * 80)

    # Without rewriting
    result_without = rag_pipeline_with_rewriting(question, use_rewriting=False)
    print(f"\nWITHOUT Query Rewriting:")
    print(f"  Search Query: {result_without['search_query']}")
    print(f"  Retrieved: {len(result_without['retrieved_ids'])} poses")
    for pose in result_without["retrieved_poses"][:3]:
        print(f"    - {pose['pose_name']} ({pose['category']})")
    print(
        f"  Answer: {result_without['answer'][:200]}..."
        if len(result_without["answer"]) > 200
        else f"  Answer: {result_without['answer']}"
    )

    # With rewriting
    result_with = rag_pipeline_with_rewriting(question, use_rewriting=True)
    print(f"\nWITH Query Rewriting:")
    print(f"  Search Query: {result_with['search_query']}")
    print(f"  Retrieved: {len(result_with['retrieved_ids'])} poses")
    for pose in result_with["retrieved_poses"][:3]:
        print(f"    - {pose['pose_name']} ({pose['category']})")
    print(
        f"  Answer: {result_with['answer'][:200]}..."
        if len(result_with["answer"]) > 200
        else f"  Answer: {result_with['answer']}"
    )

    print("\n" + "=" * 80)


SIDE-BY-SIDE COMPARISON


1. Question: What poses help with balance?
--------------------------------------------------------------------------------

WITHOUT Query Rewriting:
  Search Query: What poses help with balance?
  Retrieved: 5 poses
    - Cat-Cow Pose (standing)
    - Warrior II on One Leg (balancing)
    - Peacock Pose (balancing)
  Answer: According to the provided yoga pose information, the following poses can help with balance:

1. Warrior II on One Leg (Eka Pada Virabhadrasana II) - This pose strengthens the ankles and legs, while al...

WITH Query Rewriting:
  Search Query: What yoga asanas are beneficial for improving balance and equilibrium, particularly those that target the ankles, calves, and core muscles?
  Retrieved: 5 poses
    - Razor's Edge (balancing)
    - Eight-Limbed Pose (standing)
    - Titibhasana II (balancing)
  Answer: The following yoga poses can help improve balance: 

1. Razor's Edge (Kshura Rekha) - This advanced balancing pose strengthens the a

### Cost and Latency Analysis


In [21]:
# Measure cost and latency impact
print("COST AND LATENCY ANALYSIS\n")
print("=" * 80)

# Test on sample questions
test_sample = ground_truth.head(10)

# Without rewriting
start = time.time()
total_tokens_without = 0
for _, row in test_sample.iterrows():
    result = rag_pipeline_with_rewriting(row["question"], use_rewriting=False)
    total_tokens_without += result["tokens_used"]
time_without = time.time() - start

# With rewriting
start = time.time()
total_tokens_with = 0
for _, row in test_sample.iterrows():
    result = rag_pipeline_with_rewriting(row["question"], use_rewriting=True)
    total_tokens_with += result["tokens_used"]
time_with = time.time() - start

print(f"Tested on {len(test_sample)} questions\n")

print("WITHOUT Query Rewriting:")
print(f"  Total Time: {time_without:.2f}s")
print(f"  Avg Time per Query: {time_without/len(test_sample):.2f}s")
print(f"  Total Tokens: {total_tokens_without}")
print(f"  Avg Tokens per Query: {total_tokens_without/len(test_sample):.0f}")

print("\nWITH Query Rewriting:")
print(f"  Total Time: {time_with:.2f}s")
print(f"  Avg Time per Query: {time_with/len(test_sample):.2f}s")
print(f"  Total Tokens: {total_tokens_with}")
print(f"  Avg Tokens per Query: {total_tokens_with/len(test_sample):.0f}")

print("\nOverhead:")
time_overhead = ((time_with - time_without) / time_without) * 100
token_overhead = (
    (total_tokens_with - total_tokens_without) / total_tokens_without
) * 100
print(f"  Time Overhead: {time_overhead:+.1f}%")
print(f"  Token Overhead: {token_overhead:+.1f}%")
print("=" * 80)

COST AND LATENCY ANALYSIS

Tested on 10 questions

WITHOUT Query Rewriting:
  Total Time: 15.22s
  Avg Time per Query: 1.52s
  Total Tokens: 17374
  Avg Tokens per Query: 1737

WITH Query Rewriting:
  Total Time: 24.42s
  Avg Time per Query: 2.44s
  Total Tokens: 17630
  Avg Tokens per Query: 1763

Overhead:
  Time Overhead: +60.4%
  Token Overhead: +1.5%


## Findings and Decision

### Summary of Results

We evaluated query rewriting on 30 ground truth questions and measured its impact on:

1. **Retrieval Quality**
2. **Answer Quality**
3. **Cost and Latency**

### Quantitative Results

| Metric          | Without Rewriting | With Rewriting | Change         |
| --------------- | ----------------- | -------------- | -------------- |
| **Hit Rate**    | 83.33%            | 66.67%         | **-16.67%** ❌ |
| **MRR**         | 75.28%            | 54.17%         | **-21.11%** ❌ |
| **Avg Latency** | 1.52s             | 2.44s          | **+60.4%** ❌  |
| **Avg Tokens**  | 1,737             | 1,763          | **+1.5%**      |

### Key Observations

1. **Retrieval Quality Degraded Significantly**

   - Hit rate dropped 17 percentage points
   - MRR dropped 21 percentage points
   - Both metrics show consistent, substantial decline

2. **Query Rewriting Over-Complicates Questions**

   - Original: "What poses help with balance?"
   - Rewritten: "What yoga asanas are beneficial for improving balance and equilibrium, particularly those that target the ankles, calves, and core muscles?"
   - The rewritten query is more technical but less effective

3. **Semantic Drift Hurts Matching**

   - Rewritten queries use terms like "asanas", "equilibrium", "lumbar region"
   - Database uses simpler terms like "poses", "balance", "back"
   - Technical terminology doesn't match how poses are described

4. **Significant Latency Overhead**

   - 60% increase in response time (1.52s → 2.44s)
   - Extra LLM call adds ~900ms per query
   - Unacceptable for user experience

5. **Minimal Token Overhead**
   - Only 1.5% increase in tokens
   - Cost impact is small
   - But still wasteful given negative results

### Why Query Rewriting Failed

The hybrid search (BM25 + vector embeddings) already handles natural language queries effectively:

- **User queries are already good**: Simple, direct questions match the database well
- **LLM adds noise, not signal**: Technical terms and verbose phrasing hurt matching
- **Semantic embeddings work**: Vector search captures meaning without needing rewrites
- **BM25 handles keywords**: Text search finds exact matches in pose descriptions

### Decision

**Query rewriting will NOT be included in the production system.**

**Rationale:**

1. ❌ **Fails performance criteria**: Requires >5% improvement, got -17% hit rate and -21% MRR
2. ❌ **Unacceptable latency**: 60% slower response time degrades user experience
3. ❌ **No benefits**: Worse on every metric that matters
4. ✓ **Baseline is strong**: 83% hit rate without rewriting is already good

### Lessons Learned

1. **Not all RAG "best practices" help every system**

   - Query rewriting is commonly recommended
   - But it doesn't always improve performance
   - Always measure and validate

2. **Simple can be better than complex**

   - Users' natural language works well
   - Adding LLM "enhancement" made things worse
   - Don't add complexity without proven benefit

3. **Trust the data**
   - Clear, consistent negative results
   - No need to second-guess
   - Move on to other improvements


## Document Re-ranking Experiments

Document re-ranking reorders the initial retrieval results to improve relevance. We'll test:

1. **Vector-based re-ranking**: Use vector similarity to reorder BM25 results
2. **Measure impact on retrieval metrics** (Hit Rate, MRR)
3. **Compare answer quality** with and without re-ranking
4. **Decide whether to include in production**

### Approach

The hybrid search already combines BM25 and vector search. Re-ranking would:

- First retrieve with BM25 (fast, keyword-based)
- Then re-rank using vector similarity (semantic understanding)
- This is a two-stage approach: recall → precision


### Implement Vector-Based Re-ranking


In [22]:
def rerank_with_vector_similarity(
    query: str, retrieved_ids: List[int], top_k: int = 5
) -> List[int]:
    """
    Re-rank retrieved documents using vector similarity.

    Args:
        query: User's query
        retrieved_ids: Initial retrieval results (e.g., from BM25)
        top_k: Number of results to return after re-ranking

    Returns:
        Re-ranked list of pose IDs
    """
    if not retrieved_ids:
        return []

    # Get query embedding
    query_embedding = retrieval_system.vector_search.model.encode(
        [query], convert_to_numpy=True
    )[0]

    # Get embeddings for retrieved documents
    retrieved_indices = [
        retrieval_system.vector_search.pose_ids.index(pose_id)
        for pose_id in retrieved_ids
    ]
    retrieved_embeddings = retrieval_system.vector_search.doc_embeddings[
        retrieved_indices
    ]

    # Calculate cosine similarity
    similarities = cosine_similarity(
        query_embedding.reshape(1, -1), retrieved_embeddings
    )[0]

    # Sort by similarity and return top-k
    sorted_indices = np.argsort(similarities)[::-1][:top_k]
    return [retrieved_ids[i] for i in sorted_indices]


print("✓ Vector-based re-ranking function implemented")

✓ Vector-based re-ranking function implemented


### Test Re-ranking with Sample Query


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Test re-ranking with a sample query
test_query = "What poses help with balance?"

# Get BM25-only results
bm25_results = retrieval_system.bm25_search.search(test_query, top_k=10)

# Re-rank with vector similarity
reranked_results = rerank_with_vector_similarity(
    test_query, bm25_results, top_k=5
)

# Get hybrid search results (baseline)
hybrid_results = retrieval_system.search(test_query, top_k=5)

print(f"Query: {test_query}\n")
print("BM25 Only (top 5):")
for i, pose_id in enumerate(bm25_results[:5], 1):
    pose = pose_dict[pose_id]
    print(f"{i}. {pose['pose_name']} (ID: {pose_id})")

print("\nBM25 + Vector Re-ranking:")
for i, pose_id in enumerate(reranked_results, 1):
    pose = pose_dict[pose_id]
    print(f"{i}. {pose['pose_name']} (ID: {pose_id})")

print("\nHybrid Search (baseline):")
for i, pose_id in enumerate(hybrid_results, 1):
    pose = pose_dict[pose_id]
    print(f"{i}. {pose['pose_name']} (ID: {pose_id})")

Query: What poses help with balance?

BM25 Only (top 5):
1. Cat-Cow Pose (ID: 5)
2. Inner Spiral (ID: 172)
3. Titibhasana II (ID: 47)
4. Hanumanasana (ID: 37)
5. Warrior II on One Leg (ID: 118)

BM25 + Vector Re-ranking:
1. Warrior II on One Leg (ID: 118)
2. River Rock (ID: 199)
3. Utkatasana (ID: 73)
4. Lord of the Dance Pose (ID: 30)
5. Eka Pada Bakasana (ID: 48)

Hybrid Search (baseline):
1. Cat-Cow Pose (ID: 5)
2. Warrior II on One Leg (ID: 118)
3. Peacock Pose (ID: 33)
4. River Rock (ID: 199)
5. Warrior Pose (ID: 109)


### Implement RAG Pipeline with Re-ranking


In [ ]:
def rag_pipeline_with_reranking(
    question: str,
    use_reranking: bool = False,
    model: str = LLM_MODEL,
    top_k: int = 5,
    temperature: float = 0.3,
    max_tokens: int = 500,
) -> Dict[str, Any]:
    """
    RAG pipeline with optional document re-ranking.

    Args:
        question: User's question
        use_reranking: Whether to re-rank results with vector similarity
        model: LLM model to use
        top_k: Number of poses to retrieve
        temperature: LLM temperature
        max_tokens: Maximum tokens in response

    Returns:
        Dictionary with answer, retrieved_poses, tokens_used, response_time, etc.
    """
    start_time = time.time()

    # Step 1: Retrieve relevant poses
    if use_reranking:
        # Use hybrid search for initial retrieval (retrieve more for re-ranking)
        initial_results = retrieval_system.search(question, top_k=top_k * 2)
        # Re-rank with vector similarity to refine the top results
        retrieved_ids = rerank_with_vector_similarity(
            question, initial_results, top_k=top_k
        )
    else:
        # Use hybrid search (baseline)
        retrieved_ids = retrieval_system.search(question, top_k=top_k)

    if not retrieved_ids:
        return {
            "answer": "I couldn't find any relevant yoga poses for your question. Could you please rephrase or ask about a specific pose, category, or benefit?",
            "retrieved_poses": [],
            "retrieved_ids": [],
            "tokens_used": 0,
            "response_time_ms": int((time.time() - start_time) * 1000),
            "model": model,
            "used_reranking": use_reranking,
        }

    # Step 2: Assemble context
    context = assemble_context(retrieved_ids)

    # Step 3: Create prompt
    prompt = create_prompt(question, context)

    # Step 4: Call LLM
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        answer = response.choices[0].message.content
        tokens_used = response.usage.total_tokens
    except Exception as e:
        answer = f"Error generating response: {str(e)}"
        tokens_used = 0

    response_time_ms = int((time.time() - start_time) * 1000)

    return {
        "answer": answer,
        "retrieved_poses": [
            {
                "id": pose_id,
                "pose_name": pose_dict[pose_id]["pose_name"],
                "category": pose_dict[pose_id]["category"],
                "difficulty_level": pose_dict[pose_id]["difficulty_level"],
            }
            for pose_id in retrieved_ids
        ],
        "retrieved_ids": retrieved_ids,
        "tokens_used": tokens_used,
        "response_time_ms": response_time_ms,
        "model": model,
        "used_reranking": use_reranking,
    }


print("✓ RAG pipeline with re-ranking implemented")

✓ RAG pipeline with re-ranking implemented


### Evaluate Retrieval Quality: With vs Without Re-ranking


In [27]:
# Evaluate on ground truth dataset
print("Evaluating retrieval quality with and without re-ranking...\n")

# Sample subset for evaluation
sample_size = 30
eval_data = ground_truth.head(sample_size)

# Without re-ranking (hybrid search baseline)
print("Running WITHOUT re-ranking (hybrid search)...")
retrieved_without_reranking = []
for _, row in eval_data.iterrows():
    result = rag_pipeline_with_reranking(
        row["question"], use_reranking=False, top_k=5
    )
    retrieved_without_reranking.append(result["retrieved_ids"])

# With re-ranking
print("Running WITH re-ranking (BM25 + vector re-rank)...")
retrieved_with_reranking = []
for _, row in eval_data.iterrows():
    result = rag_pipeline_with_reranking(
        row["question"], use_reranking=True, top_k=5
    )
    retrieved_with_reranking.append(result["retrieved_ids"])

# Calculate metrics
relevant_ids = eval_data["relevant_pose_ids_list"].tolist()

metrics_without = {
    "hit_rate": calculate_hit_rate(retrieved_without_reranking, relevant_ids),
    "mrr": calculate_mrr(retrieved_without_reranking, relevant_ids),
}

metrics_with = {
    "hit_rate": calculate_hit_rate(retrieved_with_reranking, relevant_ids),
    "mrr": calculate_mrr(retrieved_with_reranking, relevant_ids),
}

print("\n" + "=" * 80)
print("RETRIEVAL QUALITY COMPARISON: RE-RANKING")
print("=" * 80)
print(f"\nEvaluated on {len(eval_data)} questions\n")

print("WITHOUT Re-ranking (Hybrid Search):")
print(f"  Hit Rate: {metrics_without['hit_rate']:.2%}")
print(f"  MRR:      {metrics_without['mrr']:.2%}")

print("\nWITH Re-ranking (BM25 → Vector):")
print(f"  Hit Rate: {metrics_with['hit_rate']:.2%}")
print(f"  MRR:      {metrics_with['mrr']:.2%}")

print("\nChange:")
hit_rate_change = metrics_with["hit_rate"] - metrics_without["hit_rate"]
mrr_change = metrics_with["mrr"] - metrics_without["mrr"]

if hit_rate_change > 0:
    print(f"  Hit Rate: {hit_rate_change:+.2%} (IMPROVEMENT ✓)")
else:
    print(f"  Hit Rate: {hit_rate_change:+.2%} (DEGRADATION ✗)")

if mrr_change > 0:
    print(f"  MRR:      {mrr_change:+.2%} (IMPROVEMENT ✓)")
else:
    print(f"  MRR:      {mrr_change:+.2%} (DEGRADATION ✗)")
print("=" * 80)

Evaluating retrieval quality with and without re-ranking...

Running WITHOUT re-ranking (hybrid search)...
Running WITH re-ranking (BM25 + vector re-rank)...

RETRIEVAL QUALITY COMPARISON: RE-RANKING

Evaluated on 30 questions

WITHOUT Re-ranking (Hybrid Search):
  Hit Rate: 83.33%
  MRR:      75.28%

WITH Re-ranking (BM25 → Vector):
  Hit Rate: 80.00%
  MRR:      68.44%

Change:
  Hit Rate: -3.33% (DEGRADATION ✗)
  MRR:      -6.83% (DEGRADATION ✗)


### Compare Answer Quality: Sample Questions


In [28]:
# Compare answers for sample questions
comparison_questions = [
    "What poses help with balance?",
    "What are the benefits of Cobra Pose?",
    "Can you recommend beginner standing poses?",
]

print("\nANSWER QUALITY COMPARISON\n")
print("=" * 80)

for i, question in enumerate(comparison_questions, 1):
    print(f"\n{i}. Question: {question}")
    print("-" * 80)

    # Without re-ranking
    result_without = rag_pipeline_with_reranking(question, use_reranking=False)
    print(f"\nWITHOUT Re-ranking (Hybrid Search):")
    print(f"  Retrieved: {len(result_without['retrieved_ids'])} poses")
    for pose in result_without["retrieved_poses"][:3]:
        print(f"    - {pose['pose_name']} ({pose['category']})")
    print(
        f"  Answer: {result_without['answer'][:200]}..."
        if len(result_without["answer"]) > 200
        else f"  Answer: {result_without['answer']}"
    )

    # With re-ranking
    result_with = rag_pipeline_with_reranking(question, use_reranking=True)
    print(f"\nWITH Re-ranking (BM25 → Vector):")
    print(f"  Retrieved: {len(result_with['retrieved_ids'])} poses")
    for pose in result_with["retrieved_poses"][:3]:
        print(f"    - {pose['pose_name']} ({pose['category']})")
    print(
        f"  Answer: {result_with['answer'][:200]}..."
        if len(result_with["answer"]) > 200
        else f"  Answer: {result_with['answer']}"
    )

    print("\n" + "=" * 80)


ANSWER QUALITY COMPARISON


1. Question: What poses help with balance?
--------------------------------------------------------------------------------

WITHOUT Re-ranking (Hybrid Search):
  Retrieved: 5 poses
    - Cat-Cow Pose (standing)
    - Warrior II on One Leg (balancing)
    - Peacock Pose (balancing)
  Answer: According to the provided yoga pose information, the following poses can help with balance:

1. Warrior II on One Leg (Eka Pada Virabhadrasana II)
2. Peacock Pose (Mayurasana)
3. River Rock (Nadi Shil...

WITH Re-ranking (BM25 → Vector):
  Retrieved: 5 poses
    - Warrior II on One Leg (balancing)
    - River Rock (balancing)
    - Utkatasana (standing)
  Answer: Based on the provided yoga pose information, the following poses can help with balance:

1. Warrior II on One Leg (Eka Pada Virabhadrasana II) - This pose strengthens the ankles and legs, while also i...


2. Question: What are the benefits of Cobra Pose?
--------------------------------------------------------

### Latency Analysis


In [29]:
# Measure latency impact
print("LATENCY ANALYSIS\n")
print("=" * 80)

# Test on sample questions
test_sample = ground_truth.head(10)

# Without re-ranking
start = time.time()
for _, row in test_sample.iterrows():
    result = rag_pipeline_with_reranking(row["question"], use_reranking=False)
time_without = time.time() - start

# With re-ranking
start = time.time()
for _, row in test_sample.iterrows():
    result = rag_pipeline_with_reranking(row["question"], use_reranking=True)
time_with = time.time() - start

print(f"Tested on {len(test_sample)} questions\n")

print("WITHOUT Re-ranking:")
print(f"  Total Time: {time_without:.2f}s")
print(f"  Avg Time per Query: {time_without/len(test_sample):.2f}s")

print("\nWITH Re-ranking:")
print(f"  Total Time: {time_with:.2f}s")
print(f"  Avg Time per Query: {time_with/len(test_sample):.2f}s")

print("\nOverhead:")
time_overhead = ((time_with - time_without) / time_without) * 100
print(f"  Time Overhead: {time_overhead:+.1f}%")
print("=" * 80)

LATENCY ANALYSIS

Tested on 10 questions

WITHOUT Re-ranking:
  Total Time: 16.70s
  Avg Time per Query: 1.67s

WITH Re-ranking:
  Total Time: 14.61s
  Avg Time per Query: 1.46s

Overhead:
  Time Overhead: -12.5%


## Re-ranking Findings and Decision

### Summary of Results

We evaluated document re-ranking on 30 ground truth questions and measured its impact on:

1. **Retrieval Quality** (Hit Rate, MRR)
2. **Answer Quality** (subjective comparison)
3. **Latency**

### Quantitative Results

| Metric          | Hybrid Search (Baseline) | Hybrid → Vector Re-rank | Change        |
| --------------- | ------------------------ | ----------------------- | ------------- |
| **Hit Rate**    | 83.33%                   | 80.00%                  | **-3.33%** ❌ |
| **MRR**         | 75.28%                   | 68.44%                  | **-6.83%** ❌ |
| **Avg Latency** | 1.67s                    | 1.46s                   | **-12.5%** ✓  |

### Key Observations

1. **Retrieval Quality Degraded**

   - Hit rate dropped 3.33 percentage points (83.33% → 80.00%)
   - MRR dropped 6.83 percentage points (75.28% → 68.44%)
   - Both metrics show consistent decline

2. **Latency Improved Slightly**

   - 12.5% faster (1.67s → 1.46s)
   - Retrieving 10 and re-ranking to 5 is slightly faster than computing hybrid scores
   - But speed gain doesn't justify quality loss

3. **Why Re-ranking Failed: Signal Degradation**

   - **The core problem**: Re-ranking throws away the BM25 signal

     - Stage 1: Hybrid search retrieves top 10 using `score = (BM25^0.4) × (Vector^0.6)`
     - Stage 2: Re-rank those 10 using ONLY vector similarity
     - Problem: We discard the carefully balanced hybrid score and replace it with pure vector score

   - **What we're doing**:

     ```
     Hybrid (top 10):  score = BM25^0.4 × Vector^0.6
     Re-ranking:       score = Vector only

     Result: We lose the BM25 component that made hybrid search effective!
     ```

4. **Why This Approach Can't Work**

   - Re-ranking is useful when you have a NEW or BETTER signal
   - Examples of valid re-ranking:

     - BM25 → LLM scoring (adds semantic understanding)
     - Bi-encoder → Cross-encoder (adds interaction modeling)
     - Fast model → Slow but better model

   - **Our case**: Hybrid → Same vector model
     - We're using the SAME vector embeddings that were already in the hybrid score
     - Re-ranking with the same signal can't add information
     - It only removes the BM25 signal, making results worse

5. **The Fundamental Issue**

   ```
   Hybrid search already optimally combines BM25 and vector:
   - Alpha=0.4 was tuned through experiments
   - This balance (40% BM25, 60% vector) maximizes hit rate and MRR

   Re-ranking changes the balance to:
   - 0% BM25, 100% vector
   - This is worse than the optimized 40/60 split
   ```

### Decision

**Document re-ranking will NOT be included in the production system.**

**Rationale:**

1. ❌ **Degrades performance**: Hit Rate -3.33%, MRR -6.83%
2. ❌ **Throws away BM25 signal**: Re-ranking with same vector model removes the keyword matching component
3. ❌ **No new information**: Using the same embeddings that were already in hybrid score
4. ✓ **Hybrid search is optimal**: The 40/60 BM25/vector balance is already tuned

### What Would Make Re-ranking Work?

Re-ranking could help if we had:

1. **LLM-based re-ranking**: Use LLM to score relevance (too expensive/slow for this use case)
2. **Cross-encoder model**: Specialized re-ranking model that considers query-document interaction
3. **Different embedding model**: A second, more powerful vector model for re-ranking
4. **Hybrid re-ranking**: Re-rank using a different BM25/vector balance (but this is just tuning alpha)

None of these are worth the added complexity for a 202-document dataset.

### Lessons Learned

1. **Re-ranking requires a new signal**

   - Re-ranking only helps if you add NEW information
   - Using the same signal that was already in the first stage can't improve results
   - In fact, it degrades results by removing other signals (BM25)

2. **Hybrid search already optimizes the signal combination**

   - The alpha=0.4 parameter was tuned through experiments
   - This represents the optimal balance of BM25 and vector for our data
   - Re-ranking with pure vector is equivalent to alpha=0 (worse)

3. **Not all "best practices" apply universally**

   - Re-ranking is recommended in RAG systems
   - But it assumes you're adding a better/different signal
   - When you're using the same signal, it can't help

4. **Small datasets favor single-stage retrieval**

   - With 202 documents, we can afford to score all documents with hybrid search
   - Two-stage retrieval is designed for millions of documents where you need a cheap first stage
   - Our dataset is small enough that single-stage fusion is optimal

5. **Always measure and validate**
   - Re-ranking seemed like it should help
   - But data showed it made things worse
   - Trust the metrics, not assumptions

### Why This Approach?

We chose vector-based re-ranking because:

1. **Fast**: No additional LLM calls (unlike LLM-based re-ranking)
2. **Semantic**: Vector similarity captures meaning
3. **Two-stage**: Separates recall (BM25) from precision (vector)
4. **Proven**: Common pattern in production RAG systems

We did NOT test LLM-based re-ranking because:

- Too slow (requires LLM call per document)
- Too expensive (tokens for each re-ranking)
- Overkill for this use case (yoga poses are well-structured)
- Vector re-ranking already failed, LLM would be even worse


## Next Steps

In subsequent experiments, we will:

- Test multiple LLM models (DeepSeek-R1, DeepSeek-V3, Qwen, Llama)
- Experiment with different prompt templates
- Evaluate answer quality using LLM-as-a-Judge
